# Explore HMRC OTS trade data before the full pull

Scratch notebook (not logged, nothing saved): check what the fields mean and how big the pull
for all minerals in `config/minerals.toml` would be, for a given start month.

In [1]:
import time
import tomllib
from pathlib import Path

import pandas as pd
import requests

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE = "https://api.uktradeinfo.com"

with open(REPO_ROOT / "config" / "minerals.toml", "rb") as f:
    minerals = tomllib.load(f)["minerals"]

def get_json(path, params=None):
    """GET one page of JSON from the HMRC API (enough for exploration)."""
    response = requests.get(f"{BASE}/{path}", params=params, timeout=60)
    response.raise_for_status()
    return response.json()

In [2]:
# What do the FlowTypeId values mean?
pd.DataFrame(get_json("FlowType")["value"])

,@odata.type,FlowTypeId,FlowTypeDescription
0,#UktiApi.FlowType,1,EU Imports ...
1,#UktiApi.FlowType,2,EU Exports ...
2,#UktiApi.FlowType,3,Non-EU Imports ...
3,#UktiApi.FlowType,4,Non-EU Exports ...


In [3]:
# One code (lithium carbonate), recent months: how many rows, and how much comes from the port split?
sample = pd.DataFrame(get_json("OTS", {"$filter": "CommodityId eq 28369100 and MonthId ge 202401"})["value"])
print(sample.shape)
print("ports:", sample["PortId"].nunique(),
      "| countries:", sample["CountryId"].nunique(),
      "| flows:", sample["FlowTypeId"].nunique())
sample.head(10)

(570, 10)
ports: 26 | countries: 46 | flows: 4


,MonthId,FlowTypeId,SuppressionIndex,CommodityId,CommoditySitcId,CountryId,PortId,Value,NetMass,SuppUnit
0,202401,1,0,28369100,52379,1,3,48068.0,2200.0,0.0
1,202401,1,0,28369100,52379,1,382,218000.0,10000.0,0.0
2,202401,1,0,28369100,52379,1,430,6475.0,0.0,0.0
3,202402,1,0,28369100,52379,1,3,6275.0,200.0,0.0
4,202403,1,0,28369100,52379,1,382,218100.0,10000.0,0.0
5,202405,1,0,28369100,52379,1,3,76800.0,6000.0,0.0
6,202406,1,0,28369100,52379,1,382,124500.0,10000.0,0.0
7,202407,1,0,28369100,52379,1,3,58050.0,5000.0,0.0
8,202407,1,0,28369100,52379,1,7,10461.0,1000.0,0.0
9,202409,1,0,28369100,52379,1,381,24501.0,2000.0,0.0


In [6]:
START_MONTH = 202201  # change to compare start years

def count_rows(codes, start=START_MONTH, batch=10):
    """Ask the API how many OTS rows match, without downloading them (in batches of codes)."""
    total = 0
    for i in range(0, len(codes), batch):
        codes_filter = " or ".join(f"CommodityId eq {int(c)}" for c in codes[i:i + batch])
        response = requests.get(f"{BASE}/OTS/$count",
                                params={"$filter": f"({codes_filter}) and MonthId ge {start}"},
                                timeout=120)
        response.raise_for_status()
        total += int(response.content.decode("utf-8-sig"))  # utf-8-sig drops the BOM the API adds
        time.sleep(0.5)  # be gentle with the API
    return total

counts = {}
for key, m in minerals.items():
    counts[key] = count_rows(m["cn8"])
    print(f"{key}: {counts[key]:,}")
    time.sleep(0.5)  # be gentle with the API

counts = pd.Series(counts).sort_values(ascending=False)
print(f"\ntotal rows from {START_MONTH}: {counts.sum():,}")
counts

aluminium: 174,450
antimony: 1,262
beryllium: 312
bismuth: 893
borates: 1,089
chromium: 4,868
cobalt: 10,937
copper: 160,244
gallium: 149
germanium: 692
graphite: 24,021
hafnium: 625
helium: 1,772
indium: 375
iridium: 1,808
iron: 84,436
lithium: 2,156
magnesite: 4,254
magnesium: 8,479
manganese: 2,853
nickel: 65,946
niobium: 959
phosphorus: 15,578
platinum: 10,099
rare_earth_elements: 8,611
rhenium: 277
rhodium: 1,499
ruthenium: 1,808
silicon: 40,798
sodium: 1,414
tantalum: 5,436
tellurium: 252
tin: 7,903
titanium: 36,326
tungsten: 9,704
uranium: 335
vanadium: 437
zinc: 14,145

total rows from 202201: 707,202


aluminium              174450
copper                 160244
iron                    84436
nickel                  65946
silicon                 40798
titanium                36326
graphite                24021
phosphorus              15578
zinc                    14145
cobalt                  10937
platinum                10099
tungsten                 9704
rare_earth_elements      8611
magnesium                8479
tin                      7903
tantalum                 5436
chromium                 4868
magnesite                4254
manganese                2853
lithium                  2156
iridium                  1808
ruthenium                1808
helium                   1772
rhodium                  1499
sodium                   1414
antimony                 1262
borates                  1089
niobium                   959
bismuth                   893
germanium                 692
hafnium                   625
vanadium                  437
indium                    375
uranium   

In [7]:
log = pd.read_json(REPO_ROOT / "data" / "raw" / "retrieval_log.jsonl", lines=True)
saved = log[log["source"].str.contains("OTS")].groupby("mineral").last()["rows"]
pd.DataFrame({"api_count": counts, "saved": saved}).assign(diff=lambda d: d["saved"] - d["api_count"])

,api_count,saved,diff
aluminium,174450,174450,0
antimony,1262,1262,0
beryllium,312,312,0
bismuth,893,893,0
borates,1089,1089,0
chromium,4868,4868,0
cobalt,10937,10937,0
copper,160244,160244,0
gallium,149,149,0
germanium,692,692,0
